# Monitoramento

Com a regra em uso num hospital, alguém teria de perceber a deriva a tempo. O X8 testa dois sinais para isso, com a
regra estática de dois cenários: a pandemia do X4 e os trimestres de 2023 a 2025 do X2. Um é o gráfico de controle do
O/E por trimestre, com limites binomiais de 95%. O outro é um martingale de potência sobre os p-valores conformais das
internações em ordem de data, um para os óbitos e outro para os sobreviventes, com alarme quando ele chega a 100 (log10
M igual a 2). Os resultados estão em `reports/exploratorios/morte/nenhuma/resultados_x8.json`.

Os alarmes aqui são descritivos. Os p-valores conformais de um cenário dividem o mesmo conjunto de calibração e não são
independentes, e sem independência o limiar de 100 não garante 1% de alarmes falsos.

In [1]:
import json
from functools import partial

import pandas as pd

from triagem_baixo_risco import config

EXPLORATORIOS = config.REPORTS_DIR / "exploratorios" / "morte" / "nenhuma"


def pct(x, casas=2):
    return f"{100 * x:.{casas}f}%".replace(".", ",")


def num(x, casas=0):
    # 1234567.891 com casas=2 vira "1.234.567,89"
    return f"{x:,.{casas}f}".replace(",", " ").replace(".", ",").replace(" ", ".")


x8 = json.loads((EXPLORATORIOS / "resultados_x8.json").read_text())


def monitoramento(res):
    oe = pd.DataFrame(res["oe"]).T
    duas, uma = partial(num, casas=2), partial(num, casas=1)
    return pd.DataFrame(
        {
            "O/E": oe["oe"].map(duas),
            "limites do O/E": oe["lim_inf"].map(duas) + " a " + oe["lim_sup"].map(duas),
            "alarme do O/E": oe["alarme"].map({True: "sim", False: "não"}),
            "log10 M, óbitos": pd.Series(res["obitos"]["log10_max_por_trimestre"]).map(uma),
            "log10 M, sobreviventes": pd.Series(res["sobreviventes"]["log10_max_por_trimestre"]).map(uma),
        }
    ).rename_axis("trimestre")

## Pandemia

As colunas de log10 M trazem o maior valor do martingale dentro de cada trimestre.

In [2]:
monitoramento(x8["x4"])

,O/E,limites do O/E,alarme do O/E,"log10 M, óbitos","log10 M, sobreviventes"
trimestre,,,,,
2020T1,"0,93","0,94 a 1,06",sim,"-0,3","0,0"
2020T2,"1,06","0,94 a 1,06",não,"-0,3","11,5"
2020T3,"1,18","0,94 a 1,06",sim,"-1,2","33,7"
2020T4,"1,14","0,94 a 1,06",sim,"-1,4","42,8"
2021T1,"1,51","0,95 a 1,06",sim,"0,4","60,9"
2021T2,"1,28","0,95 a 1,05",sim,"6,4","114,4"
2021T3,"1,16","0,95 a 1,05",sim,"8,6","140,9"
2021T4,"1,05","0,95 a 1,06",não,"8,0","139,3"


O gráfico do O/E deu alarme já em 2020T1, com O/E de 0,93, abaixo do limite, e depois em todos os trimestres de 2020T3 a
2021T3, quando o modelo passou a subestimar os óbitos. O martingale dos sobreviventes disparou em 2020T2 e chegou a
log10 M de 140,9 em 2021T3. O dos óbitos, que acompanha os escores de que G1 depende, só passou de 2 em 2021T2, o
trimestre em que G2 falhou no X4.

## De 2023 a 2025

In [3]:
monitoramento(x8["x2"])

,O/E,limites do O/E,alarme do O/E,"log10 M, óbitos","log10 M, sobreviventes"
trimestre,,,,,
2023T1,"0,97","0,95 a 1,05",não,"-0,2","0,3"
2023T2,"0,93","0,95 a 1,05",sim,"-1,1","-5,5"
2023T3,"0,92","0,95 a 1,05",sim,"-0,5","-9,9"
2023T4,"0,97","0,95 a 1,05",não,"-0,8","-17,5"
2024T1,"0,89","0,95 a 1,05",sim,"0,0","-23,2"
2024T2,"0,85","0,95 a 1,05",sim,"0,7","-29,1"
2024T3,"0,90","0,95 a 1,05",sim,"-0,3","-35,8"
2024T4,"0,83","0,95 a 1,05",sim,"-1,1","-39,6"
2025T1,"0,81","0,95 a 1,05",sim,"-1,8","-36,6"


Aqui o O/E ficou abaixo do limite inferior em dez dos doze trimestres, e em todos a partir de 2024T1. Nenhum martingale
disparou. O dos óbitos chegou no máximo a 0,7, em 2024T2, e em 2024T1, o trimestre da violação de G1 no X2, ficou em
0,0; o dos sobreviventes ficou negativo depois de 2023T1.

In [4]:
pd.DataFrame(
    {
        cenario: {
            "martingale dos óbitos": r["obitos"]["alarme_no_trimestre"] or "nenhum",
            "martingale dos sobreviventes": r["sobreviventes"]["alarme_no_trimestre"] or "nenhum",
            "primeiro alarme do O/E": next((q for q, v in r["oe"].items() if v["alarme"]), "nenhum"),
        }
        for cenario, r in {"pandemia (X4)": x8["x4"], "2023-2025 (X2)": x8["x2"]}.items()
    }
).T

,martingale dos óbitos,martingale dos sobreviventes,primeiro alarme do O/E
pandemia (X4),2021T2,2020T2,2020T1
2023-2025 (X2),nenhum,nenhum,2023T2


O gráfico do O/E reage ao nível da mortalidade, que mudou nos dois cenários, e alarmou cedo nos dois. O martingale dos
óbitos só disparou na pandemia, em 2021T2, mais de um ano depois do primeiro alarme do O/E, e em 2023-2025 não viu o
trimestre da dengue.